# Pipeline RAG con datapizza-ai

Questo notebook guida passo passo nella costruzione di una pipeline RAG (Retrieval-Augmented Generation) per interrogare documenti PDF.

## Cos'è una pipeline RAG?

Una pipeline RAG permette di:
1. **Indicizzare documenti** (ingestion): estrarre testo, dividerlo in chunk, convertirlo in vettori
2. **Interrogare i documenti** (retrieval): trovare i chunk più rilevanti per una domanda
3. **Generare risposte** (generation): usare un LLM per rispondere basandosi sui chunk trovati

## Architettura

```
INGESTION:   PDF → Parser → Splitter → Embedder → Vector Store

RETRIEVAL:   Query → Embedder → Search → Chunks → LLM → Risposta
```


## 1. Setup e configurazione

Prima di tutto, carichiamo le variabili d'ambiente e definiamo le costanti di configurazione.


In [20]:
import os
from dotenv import load_dotenv

# Carica le variabili d'ambiente dal file .env
load_dotenv()

# Verifica che la API key sia presente
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY non trovata. Crea un file .env con: OPENAI_API_KEY=sk-...")

print("API key caricata correttamente")


API key caricata correttamente


In [26]:
# Configurazione della pipeline
COLLECTION_NAME = "ducati_docs"      # Nome della collection nel vector store
QDRANT_PATH = "../qdrant_data"       # Cartella per la persistenza locale
EMBEDDING_MODEL = "text-embedding-3-small"  # Modello OpenAI per gli embeddings
EMBEDDING_DIM = 1536                 # Dimensione dei vettori
LLM_MODEL = "gpt-5.1"                # Modello per la generazione delle risposte
PDF_PATH = "Datapizza/Ducati/data/Ducati_Overview.pdf"  # Documento da indicizzare

print(f"Collection: {COLLECTION_NAME}")
print(f"Storage: {QDRANT_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model: {LLM_MODEL}")


Collection: ducati_docs
Storage: ../qdrant_data
Embedding model: text-embedding-3-small
LLM model: gpt-5.1


## 2. Creazione del vector store

Il vector store è il database dove salviamo i vettori (embeddings) dei chunk di testo.

Usiamo **Qdrant** con persistenza locale: i dati vengono salvati in una cartella e rimangono disponibili tra le sessioni.


In [22]:
from datapizza.vectorstores.qdrant import QdrantVectorstore
from datapizza.core.vectorstore import VectorConfig

# Crea il vector store con persistenza locale
# location=None bypassa un check interno, path specifica la cartella di storage
vectorstore = QdrantVectorstore(location=None, path=QDRANT_PATH)

# Crea la collection con la configurazione dei vettori
# - name: nome del modello di embedding usato
# - dimensions: dimensione dei vettori (1536 per text-embedding-3-small)
vectorstore.create_collection(
    COLLECTION_NAME,
    vector_config=[
        VectorConfig(name=EMBEDDING_MODEL, dimensions=EMBEDDING_DIM)
    ]
)

print(f"Vector store creato in: {os.path.abspath(QDRANT_PATH)}")


Vector store creato in: /home/mcalcaterra/Documenti/qdrant_data


## 3. Pipeline di ingestion

La pipeline di ingestion trasforma un documento PDF in chunk vettoriali salvati nel database.

I componenti sono:
1. **DoclingParser**: estrae testo strutturato dal PDF (testo, tabelle, immagini)
2. **NodeSplitter**: divide il documento in chunk di massimo 1000 caratteri
3. **ChunkEmbedder**: converte ogni chunk in un vettore usando OpenAI embeddings


In [23]:
from datapizza.embedders import ChunkEmbedder
from datapizza.embedders.openai import OpenAIEmbedder
from datapizza.modules.parsers.docling import DoclingParser
from datapizza.modules.splitters import NodeSplitter
from datapizza.pipeline import IngestionPipeline

# Crea l'embedder che convertirà i chunk in vettori
embedder = OpenAIEmbedder(
    api_key=api_key,
    model_name=EMBEDDING_MODEL,
)

# Crea la pipeline di ingestion
# I moduli vengono eseguiti in sequenza: parser → splitter → embedder
pipeline = IngestionPipeline(
    modules=[
        DoclingParser(),                 # Estrae testo dal PDF
        NodeSplitter(max_char=1000),     # Divide in chunk di max 1000 caratteri  
        ChunkEmbedder(client=embedder),  # Genera embeddings per ogni chunk
    ],
    vector_store=vectorstore,
    collection_name=COLLECTION_NAME
)

print("Pipeline di ingestion creata")


Pipeline di ingestion creata


### Esecuzione dell'ingestion

Ora eseguiamo la pipeline sul documento PDF. Questo passaggio:
- Legge e analizza il PDF
- Lo divide in chunk
- Genera gli embeddings
- Salva tutto nel vector store

**Nota**: questa operazione può richiedere qualche minuto, specialmente il parsing del PDF.


In [25]:
!ls

Apollo-MCP    datapizzaAI-RAG	  MyBlog	   volley-scout-touch
call-up-crew  FAME		  PersonaProjects
CREDEM	      food-ai-meal-maker  pesi-avis
Datapizza     Mondadori		  VibeTalking


In [27]:
# Verifica che il file esista
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"File non trovato: {PDF_PATH}")

print(f"Elaborazione documento: {PDF_PATH}")
print("Questo può richiedere qualche minuto...")

# Esegui la pipeline
# metadata viene salvato insieme ai chunk per tracciare la fonte
pipeline.run(PDF_PATH, metadata={"source": os.path.basename(PDF_PATH)})

print("\nIngestion completata!")


Elaborazione documento: Datapizza/Ducati/data/Ducati_Overview.pdf
Questo può richiedere qualche minuto...
INFO     [2025-12-03 23:23:39 - datapizza.pipeline.pipeline:22] Running component DoclingParser


2025-12-03 23:23:39,998 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-03 23:23:40,091 - INFO - Going to convert document batch...
2025-12-03 23:23:40,092 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e5ef4f38584e392cbc05a9f53981701a
2025-12-03 23:23:40,100 - INFO - Loading plugin 'docling_defaults'
2025-12-03 23:23:40,102 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-03 23:23:40,110 - INFO - Loading plugin 'docling_defaults'
2025-12-03 23:23:40,115 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-03 23:23:40,333 - INFO - Accelerator device: 'cpu'
2025-12-03 23:23:42,558 - INFO - Loading plugin 'docling_defaults'
2025-12-03 23:23:42,563 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2025-12-03 23:23:42,576 - INFO - Accelerator device: 'cpu'
2025-12-03 23:23:43,276 - INFO - Loading plugin 'docling_defaults'
20

INFO     [2025-12-03 23:25:00 - datapizza.pipeline.pipeline:22] Running component NodeSplitter
INFO     [2025-12-03 23:25:00 - datapizza.pipeline.pipeline:22] Running component ChunkEmbedder


2025-12-03 23:25:02,162 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



Ingestion completata!


## 4. Retrieval: ricerca semantica

Ora che i documenti sono indicizzati, possiamo cercare chunk rilevanti per una query.

Il processo è:
1. Convertiamo la query in un vettore (embedding)
2. Cerchiamo i vettori più simili nel database
3. Recuperiamo i chunk di testo corrispondenti


In [32]:
# Query di esempio
query = "Quali sono i reparti principali definiti dal cliente??"

# Genera l'embedding della query
# Usiamo lo stesso modello usato per i documenti
query_embedding = embedder.embed(query)

print(f"Query: {query}")
print(f"Embedding generato: vettore di {len(query_embedding)} dimensioni")


2025-12-03 23:26:08,102 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Query: Quali sono i reparti principali definiti dal cliente??
Embedding generato: vettore di 1536 dimensioni


In [33]:
# Cerca i chunk più simili nel vector store
# k=5 significa che recuperiamo i 5 chunk più rilevanti
results = vectorstore.search(
    query_vector=query_embedding,
    collection_name=COLLECTION_NAME,
    k=5
)

print(f"Trovati {len(results)} chunk rilevanti:\n")

for i, chunk in enumerate(results, 1):
    print(f"--- Chunk {i} ---")
    print(chunk.text[:300] + "..." if len(chunk.text) > 300 else chunk.text)
    print()


Trovati 5 chunk rilevanti:

--- Chunk 1 ---
Luogo: Borgo Panigale Dipendenti IT: ~40 Consulenti gestiti: ~300 Nota: anche più tecnici sono grandi Project Manager che coordinano team esterni. 2 Persone Chiave Andrea Spina CIO, responsabile R&D; e dati. In Ducati da 20 anni. Spinge sulla cultura Al e sulla trasformazione digitale. Massimiliano ...

--- Chunk 2 ---
- Supervisione; policy; processi; standardizzazione.
 - Strumenti Al in Ducati
 - Copilot Chat (web) per tutti.
 - Copilot Pro (15 persone) .
 - Copilot Studio (5 persone) .
 - Cultura e Formazione AI



--- Chunk 3 ---
- Chatbot; agenti, RAG, multi-agente.
 - 2x4 ore, due classi: applicativilgovernance e infraldatilsecurity .
 - Teoria con minima pratica.
 - 6 Obiettivi Specifici
 - Diffondere cultura Al e comprensione trasversale.
 - Mappare use case attuali e futuri.
 - Collegare obiettivi aziendali con opportun...

--- Chunk 4 ---
1. Contesto Generale Età media team IT: 45 anni

--- Chunk 5 ---
- Rimozione timori + 3h di pr

## 5. Generation: risposta con LLM

Ora usiamo un LLM per generare una risposta basata sui chunk recuperati.

Il prompt è strutturato per:
- Fornire il contesto (i chunk trovati)
- Istruire l'LLM a rispondere solo in base al contesto
- Evitare che inventi informazioni non presenti nel documento


In [34]:
from datapizza.clients.openai import OpenAIClient

# Crea il client LLM per la generazione delle risposte
llm = OpenAIClient(
    model=LLM_MODEL,
    api_key=api_key
)

# Costruisci il contesto unendo i chunk trovati
context = "\n---\n".join([chunk.text for chunk in results])

# Crea il prompt con contesto e istruzioni
prompt = f"""Sei un assistente tecnico. Rispondi alla domanda dell'utente basandoti ESCLUSIVAMENTE sul contesto fornito.
Se l'informazione non è presente nel contesto, rispondi: "Non ho trovato questa informazione nel documento."
Non inventare informazioni.

CONTESTO:
{context}

DOMANDA: {query}

RISPOSTA:"""

print("Prompt creato. Lunghezza:", len(prompt), "caratteri")


Prompt creato. Lunghezza: 2342 caratteri


In [35]:
# Genera la risposta
response = llm.invoke(prompt)

print("Domanda:", query)
print("\nRisposta:")
print(response.text)


2025-12-03 23:26:23,565 - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


Domanda: Quali sono i reparti principali definiti dal cliente??

Risposta:
I reparti principali definiti dal cliente sono:

- Mondo Applicativo  
- Solution Design  
- Mondo Infrastruttura  
- Mondo Dati  
- Mondo Legal Security  

All’interno del Mondo Applicativo sono inoltre indicate le seguenti aree:  
- Logistica & Planning  
- R&D  
- Sales  
- ERP  
- Finance  
- Digital  
- HR
